In [2]:
# -*- coding: utf-8 -*-
"""Laboratory Work No. 8: Multiple Precision Integer Arithmetic"""

import math

In [3]:
def str_to_digits(s):
    """Convert string of digits to list of integers"""
    return [int(ch) for ch in s]

def digits_to_str(digits):
    """Convert list of digits to string"""
    return ''.join([str(d) for d in digits])

In [4]:
def add(u, v, b=10):
    """
    Algorithm 1: Addition of non-negative integers
    u, v: strings of digits (same length)
    b: base (default 10)
    Returns: sum as string
    """
    # Convert strings to lists of digits
    u_digits = str_to_digits(u)
    v_digits = str_to_digits(v)
    n = len(u_digits)

    # Initialize
    j = n - 1  # index from rightmost digit (0-based)
    k = 0      # carry
    w_digits = [0] * n

    # Process digits from right to left
    while j >= 0:
        total = u_digits[j] + v_digits[j] + k
        w_digits[j] = total % b
        k = total // b
        j -= 1

    # Add final carry if needed
    if k > 0:
        result = str(k) + digits_to_str(w_digits)
    else:
        result = digits_to_str(w_digits)

    return result

In [5]:
def subtract(u, v, b=10):
    """
    Algorithm 2: Subtraction of non-negative integers (u > v)
    u, v: strings of digits (same length, u > v)
    b: base (default 10)
    Returns: difference as string
    """
    # Convert strings to lists of digits
    u_digits = str_to_digits(u)
    v_digits = str_to_digits(v)
    n = len(u_digits)

    # Initialize
    j = n - 1  # index from rightmost digit
    k = 0      # borrow
    w_digits = [0] * n

    # Process digits from right to left
    while j >= 0:
        diff = u_digits[j] - v_digits[j] + k
        if diff < 0:
            diff += b
            k = -1  # borrow from next digit
        else:
            k = 0
        w_digits[j] = diff
        j -= 1

    # Remove leading zeros
    result = digits_to_str(w_digits).lstrip('0')
    if not result:
        result = '0'

    return result

In [6]:
def multiply_column(u, v, b=10):
    """
    Algorithm 3: Multiplication by column
    u, v: strings of digits
    b: base (default 10)
    Returns: product as string
    """
    u_digits = str_to_digits(u)
    v_digits = str_to_digits(v)
    n = len(u_digits)
    m = len(v_digits)

    # Initialize result array
    w_digits = [0] * (m + n)

    # Process each digit of v from right to left
    for j in range(m-1, -1, -1):
        if v_digits[j] == 0:
            continue

        k = 0  # carry
        # Multiply by each digit of u
        for i in range(n-1, -1, -1):
            product = u_digits[i] * v_digits[j] + w_digits[i + j + 1] + k
            w_digits[i + j + 1] = product % b
            k = product // b

        w_digits[j] = k  # final carry for this digit

    # Remove leading zeros
    result = digits_to_str(w_digits).lstrip('0')
    if not result:
        result = '0'

    return result

In [7]:
def multiply_fast(u, v, b=10):
    """
    Algorithm 4: Fast multiplication
    u, v: strings of digits
    b: base (default 10)
    Returns: product as string
    """
    u_digits = str_to_digits(u)
    v_digits = str_to_digits(v)
    n = len(u_digits)
    m = len(v_digits)

    # Initialize result array
    w_digits = [0] * (m + n)
    t = 0

    # Process all diagonals
    for s in range(m + n):
        # Sum products along diagonal s
        for i in range(s + 1):
            u_idx = n - 1 - i
            v_idx = m - 1 - (s - i)
            if 0 <= u_idx < n and 0 <= v_idx < m:
                t += u_digits[u_idx] * v_digits[v_idx]

        # Store result digit and keep carry
        w_digits[m + n - 1 - s] = t % b
        t = t // b

    # Remove leading zeros
    result = digits_to_str(w_digits).lstrip('0')
    if not result:
        result = '0'

    return result

In [11]:
def divide(u, v, b=10):
    """
    Long division algorithm for multi-digit integers
    u, v: strings of digits
    b: base (default 10)
    Returns: (quotient, remainder) as strings
    """
    # Handle division by zero
    if v == "0":
        raise ValueError("Division by zero")

    # If u < v, quotient is 0, remainder is u
    if len(u) < len(v) or (len(u) == len(v) and u < v):
        return "0", u

    # Convert to lists of digits
    u_digits = [int(d) for d in u]
    v_digits = [int(d) for d in v]
    v_len = len(v_digits)

    quotient = []
    remainder = 0

    # Process each digit of u
    i = 0
    while i < len(u_digits):
        # Bring down next digit
        remainder = remainder * b + u_digits[i]

        # Find quotient digit
        q_digit = 0
        while remainder >= int(v):
            remainder -= int(v)
            q_digit += 1

        quotient.append(q_digit)
        i += 1

    # Remove leading zeros from quotient
    while len(quotient) > 1 and quotient[0] == 0:
        quotient.pop(0)

    # Convert to strings
    quotient_str = ''.join(str(d) for d in quotient)
    remainder_str = str(remainder)

    return quotient_str, remainder_str

In [8]:
def test_algorithms():
    """Test all algorithms with examples"""

    print("=" * 60)
    print("TESTING MULTIPLE PRECISION ARITHMETIC ALGORITHMS")
    print("=" * 60)

    # Test 1: Addition
    print("\n1. ADDITION")
    u1, v1 = "12345", "56789"
    result1 = add(u1, v1)
    expected1 = str(int(u1) + int(v1))
    print(f"{u1} + {v1} = {result1}")
    print(f"Expected: {expected1}")
    print(f"Correct: {result1 == expected1}")

    # Test 2: Subtraction
    print("\n2. SUBTRACTION")
    u2, v2 = "56789", "12345"
    result2 = subtract(u2, v2)
    expected2 = str(int(u2) - int(v2))
    print(f"{u2} - {v2} = {result2}")
    print(f"Expected: {expected2}")
    print(f"Correct: {result2 == expected2}")

    # Test 3: Multiplication (column)
    print("\n3. MULTIPLICATION (COLUMN)")
    u3, v3 = "123", "45"
    result3 = multiply_column(u3, v3)
    expected3 = str(int(u3) * int(v3))
    print(f"{u3} × {v3} = {result3}")
    print(f"Expected: {expected3}")
    print(f"Correct: {result3 == expected3}")

    # Test 4: Multiplication (fast)
    print("\n4. MULTIPLICATION (FAST)")
    u4, v4 = "12345", "6789"
    result4 = multiply_fast(u4, v4)
    expected4 = str(int(u4) * int(v4))
    print(f"{u4} × {v4} = {result4}")
    print(f"Expected: {expected4}")
    print(f"Correct: {result4 == expected4}")

    # Test 5: Division
    print("\n5. DIVISION")
    u5, v5 = "100", "7"
    q5, r5 = divide(u5, v5)
    expected_q = str(int(u5) // int(v5))
    expected_r = str(int(u5) % int(v5))
    print(f"{u5} ÷ {v5} = {q5} remainder {r5}")
    print(f"Expected: {expected_q} remainder {expected_r}")
    print(f"Correct: {q5 == expected_q and r5 == expected_r}")

    print("\n" + "=" * 60)

In [12]:
def main():
    """Main program function"""

    print("LABORATORY WORK No. 8")
    print("Multiple Precision Integer Arithmetic")
    print()
    print("Algorithms implemented:")
    print("  1. Addition")
    print("  2. Subtraction")
    print("  3. Multiplication (column)")
    print("  4. Multiplication (fast)")
    print("  5. Division")
    print()

    # Run tests
    test_algorithms()

    print("\nProgram finished")

if __name__ == "__main__":
    main()

LABORATORY WORK No. 8
Multiple Precision Integer Arithmetic

Algorithms implemented:
  1. Addition
  2. Subtraction
  3. Multiplication (column)
  4. Multiplication (fast)
  5. Division

TESTING MULTIPLE PRECISION ARITHMETIC ALGORITHMS

1. ADDITION
12345 + 56789 = 69134
Expected: 69134
Correct: True

2. SUBTRACTION
56789 - 12345 = 44444
Expected: 44444
Correct: True

3. MULTIPLICATION (COLUMN)
123 × 45 = 5535
Expected: 5535
Correct: True

4. MULTIPLICATION (FAST)
12345 × 6789 = 83810205
Expected: 83810205
Correct: True

5. DIVISION
100 ÷ 7 = 14 remainder 2
Expected: 14 remainder 2
Correct: True


Program finished
